In [1]:
# =============================================================================
# NOTEBOOK 09 — CELL 1
# BASELINE EXPERIMENT SETUP AND DATA VALIDATION
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata
import json
import os
import platform
import random
import sys
import warnings

import numpy as np
import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")


# =============================================================================
# 1. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

VLM_DATASET_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vlm_dataset"
)

SPLIT_DIR = VLM_DATASET_DIR / "splits"
TEST_DATASET_PATH = SPLIT_DIR / "test.csv"

BASELINE_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "vlm_baselines"
)

OUTPUT_ROOT = BASELINE_ROOT

CONFIG_DIR = BASELINE_ROOT / "configuration"
PREDICTION_DIR = BASELINE_ROOT / "predictions"
METRIC_DIR = BASELINE_ROOT / "metrics"
VALIDATION_DIR = BASELINE_ROOT / "validation"
LOG_DIR = BASELINE_ROOT / "logs"
CACHE_DIR = BASELINE_ROOT / "cache"

for directory in [
    BASELINE_ROOT,
    CONFIG_DIR,
    PREDICTION_DIR,
    METRIC_DIR,
    VALIDATION_DIR,
    LOG_DIR,
    CACHE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


print("=" * 80)
print("NOTEBOOK 09: ROADFLOOD-VLM BASELINE EVALUATION")
print("=" * 80)

print("\nPROJECT PATHS")
print("-" * 80)
print(f"Project root      : {PROJECT_ROOT}")
print(f"Test dataset      : {TEST_DATASET_PATH}")
print(f"Baseline outputs  : {BASELINE_ROOT}")


# =============================================================================
# 2. REPRODUCIBILITY CONFIGURATION
# =============================================================================

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("\nREPRODUCIBILITY")
print("-" * 80)
print(f"Random seed : {RANDOM_SEED}")


# =============================================================================
# 3. PACKAGE AND HARDWARE DISCOVERY
# =============================================================================

def get_package_version(package_name):
    """Return an installed package version or an empty string."""

    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return ""


package_versions = {
    "python": platform.python_version(),
    "pandas": get_package_version("pandas"),
    "numpy": get_package_version("numpy"),
    "pillow": get_package_version("Pillow"),
    "torch": get_package_version("torch"),
    "transformers": get_package_version("transformers"),
    "accelerate": get_package_version("accelerate"),
    "qwen-vl-utils": get_package_version("qwen-vl-utils"),
    "sentence-transformers": get_package_version(
        "sentence-transformers"
    ),
    "rouge-score": get_package_version("rouge-score"),
    "bert-score": get_package_version("bert-score"),
    "evaluate": get_package_version("evaluate"),
}

try:
    import torch

    TORCH_AVAILABLE = True
    CUDA_AVAILABLE = torch.cuda.is_available()

    if CUDA_AVAILABLE:
        DEVICE = "cuda"
        GPU_NAME = torch.cuda.get_device_name(0)
        GPU_COUNT = torch.cuda.device_count()
        GPU_MEMORY_GB = round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2,
        )
    else:
        DEVICE = "cpu"
        GPU_NAME = ""
        GPU_COUNT = 0
        GPU_MEMORY_GB = 0.0

except ImportError:
    TORCH_AVAILABLE = False
    CUDA_AVAILABLE = False
    DEVICE = "cpu"
    GPU_NAME = ""
    GPU_COUNT = 0
    GPU_MEMORY_GB = 0.0


print("\nSOFTWARE ENVIRONMENT")
print("-" * 80)

for package_name, version in package_versions.items():
    display_version = version if version else "NOT INSTALLED"
    print(f"{package_name:<24}: {display_version}")

print("\nHARDWARE")
print("-" * 80)
print(f"Operating system : {platform.platform()}")
print(f"Processor        : {platform.processor()}")
print(f"PyTorch detected : {TORCH_AVAILABLE}")
print(f"CUDA available   : {CUDA_AVAILABLE}")
print(f"Selected device  : {DEVICE}")
print(f"GPU count        : {GPU_COUNT}")
print(f"GPU name         : {GPU_NAME or 'None'}")
print(f"GPU memory       : {GPU_MEMORY_GB:.2f} GB")


# =============================================================================
# 4. MODEL CONFIGURATION
# =============================================================================

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

# Conservative generation settings for deterministic evaluation.
GENERATION_CONFIG = {
    "max_new_tokens": 384,
    "do_sample": False,
    "temperature": None,
    "top_p": None,
    "repetition_penalty": 1.0,
}

# Image-size controls reduce memory consumption during baseline inference.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

print("\nMODEL CONFIGURATION")
print("-" * 80)
print(f"Model ID       : {MODEL_ID}")
print(f"Device         : {DEVICE}")
print(f"Maximum tokens : {GENERATION_CONFIG['max_new_tokens']}")
print(f"Sampling       : {GENERATION_CONFIG['do_sample']}")
print(f"Minimum pixels : {MIN_PIXELS:,}")
print(f"Maximum pixels : {MAX_PIXELS:,}")


# =============================================================================
# 5. BASELINE EXPERIMENT DEFINITIONS
# =============================================================================

BASELINE_CONFIGURATIONS = {
    "B0_text_only": {
        "display_name": "Text-only",
        "use_sentinel2": False,
        "use_sentinel1": False,
        "use_structured_context": False,
        "description": (
            "Uses only the original natural-language instruction."
        ),
    },
    "B1_s2_only": {
        "display_name": "Sentinel-2 only",
        "use_sentinel2": True,
        "use_sentinel1": False,
        "use_structured_context": False,
        "description": (
            "Uses the Sentinel-2 true-color image and instruction."
        ),
    },
    "B2_s1_only": {
        "display_name": "Sentinel-1 only",
        "use_sentinel2": False,
        "use_sentinel1": True,
        "use_structured_context": False,
        "description": (
            "Uses the Sentinel-1 radar visualization and instruction."
        ),
    },
    "B3_s2_s1": {
        "display_name": "Sentinel-2 + Sentinel-1",
        "use_sentinel2": True,
        "use_sentinel1": True,
        "use_structured_context": False,
        "description": (
            "Uses both image products and the original instruction."
        ),
    },
    "B4_s2_s1_transport": {
        "display_name": (
            "Sentinel-2 + Sentinel-1 + transportation context"
        ),
        "use_sentinel2": True,
        "use_sentinel1": True,
        "use_structured_context": True,
        "description": (
            "Uses both images, the instruction, and structured "
            "transportation-flood metadata."
        ),
    },
}

baseline_configuration_df = pd.DataFrame(
    [
        {
            "baseline_id": baseline_id,
            **configuration,
        }
        for baseline_id, configuration
        in BASELINE_CONFIGURATIONS.items()
    ]
)

print("\nBASELINE CONFIGURATIONS")
print("-" * 80)

print(
    baseline_configuration_df[
        [
            "baseline_id",
            "display_name",
            "use_sentinel2",
            "use_sentinel1",
            "use_structured_context",
        ]
    ].to_string(index=False)
)


# =============================================================================
# 6. LOAD TEST DATASET
# =============================================================================

if not TEST_DATASET_PATH.exists():
    raise FileNotFoundError(
        "The Notebook 08 test split was not found.\n"
        f"Expected path: {TEST_DATASET_PATH}"
    )

test_df = pd.read_csv(
    TEST_DATASET_PATH,
    keep_default_na=False,
)

required_columns = [
    "instruction_id",
    "scene_id",
    "task_family_normalized",
    "instruction_text",
    "response_text",
    "training_s2_relative_path",
    "training_s1_relative_path",
    "flood_burden_normalized",
    "disruption_normalized",
    "reliability_normalized",
    "split",
    "training_ready",
]

missing_columns = [
    column
    for column in required_columns
    if column not in test_df.columns
]

if missing_columns:
    raise ValueError(
        "The test dataset is missing required columns: "
        + ", ".join(missing_columns)
    )

test_df["training_ready"] = (
    test_df["training_ready"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(
        {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        }
    )
)

print("\nTEST DATASET")
print("-" * 80)
print(f"Records       : {len(test_df):,}")
print(f"Scenes        : {test_df['scene_id'].nunique():,}")
print(
    f"Task families : "
    f"{test_df['task_family_normalized'].nunique():,}"
)
print(
    f"Ready records : "
    f"{test_df['training_ready'].sum():,}"
)


# =============================================================================
# 7. RESOLVE AND VALIDATE IMAGE PATHS
# =============================================================================

def resolve_dataset_path(stored_path):
    """Resolve an image path stored relative to the VLM dataset directory."""

    path = Path(str(stored_path).strip())

    if path.is_absolute():
        return path

    return VLM_DATASET_DIR / path


def validate_rgb_image(image_path):
    """Validate that a model input is a readable RGB image."""

    result = {
        "exists": False,
        "readable": False,
        "format": "",
        "mode": "",
        "width": 0,
        "height": 0,
        "error": "",
    }

    try:
        image_path = Path(image_path)
        result["exists"] = image_path.exists()

        if not result["exists"]:
            result["error"] = "File does not exist."
            return result

        with Image.open(image_path) as image:
            image.load()

            result["readable"] = True
            result["format"] = str(image.format)
            result["mode"] = str(image.mode)
            result["width"] = int(image.width)
            result["height"] = int(image.height)

        return result

    except Exception as exc:
        result["error"] = str(exc)
        return result


test_df["s2_absolute_path"] = (
    test_df["training_s2_relative_path"]
    .apply(resolve_dataset_path)
    .astype(str)
)

test_df["s1_absolute_path"] = (
    test_df["training_s1_relative_path"]
    .apply(resolve_dataset_path)
    .astype(str)
)

unique_test_scenes_df = (
    test_df[
        [
            "scene_id",
            "s2_absolute_path",
            "s1_absolute_path",
        ]
    ]
    .drop_duplicates(subset=["scene_id"])
    .sort_values("scene_id")
    .reset_index(drop=True)
)

image_validation_records = []

print("\nTEST IMAGE VALIDATION")
print("-" * 80)

for row in unique_test_scenes_df.itertuples(index=False):
    s2_result = validate_rgb_image(
        row.s2_absolute_path
    )

    s1_result = validate_rgb_image(
        row.s1_absolute_path
    )

    scene_images_valid = bool(
        s2_result["exists"]
        and s2_result["readable"]
        and s2_result["format"] == "PNG"
        and s2_result["mode"] == "RGB"
        and s2_result["width"] == 512
        and s2_result["height"] == 512
        and s1_result["exists"]
        and s1_result["readable"]
        and s1_result["format"] == "PNG"
        and s1_result["mode"] == "RGB"
        and s1_result["width"] == 512
        and s1_result["height"] == 512
    )

    image_validation_records.append(
        {
            "scene_id": row.scene_id,
            "s2_path": row.s2_absolute_path,
            "s2_exists": s2_result["exists"],
            "s2_readable": s2_result["readable"],
            "s2_format": s2_result["format"],
            "s2_mode": s2_result["mode"],
            "s2_width": s2_result["width"],
            "s2_height": s2_result["height"],
            "s2_error": s2_result["error"],
            "s1_path": row.s1_absolute_path,
            "s1_exists": s1_result["exists"],
            "s1_readable": s1_result["readable"],
            "s1_format": s1_result["format"],
            "s1_mode": s1_result["mode"],
            "s1_width": s1_result["width"],
            "s1_height": s1_result["height"],
            "s1_error": s1_result["error"],
            "scene_images_valid": scene_images_valid,
        }
    )

    print(
        f"{row.scene_id:<20} "
        f"| S2 valid: {s2_result['readable']} "
        f"| S1 valid: {s1_result['readable']} "
        f"| Scene ready: {scene_images_valid}"
    )

image_validation_df = pd.DataFrame(
    image_validation_records
)


# =============================================================================
# 8. DATASET DISTRIBUTION TABLES
# =============================================================================

task_family_distribution_df = (
    test_df.groupby(
        "task_family_normalized",
        as_index=False,
    )
    .agg(
        record_count=("instruction_id", "count"),
        scene_count=("scene_id", "nunique"),
    )
    .rename(
        columns={
            "task_family_normalized": "task_family",
        }
    )
)

reliability_distribution_df = (
    test_df.groupby(
        "reliability_normalized",
        as_index=False,
    )
    .agg(
        record_count=("instruction_id", "count"),
        scene_count=("scene_id", "nunique"),
    )
    .rename(
        columns={
            "reliability_normalized": "reliability",
        }
    )
)

flood_burden_distribution_df = (
    test_df.groupby(
        "flood_burden_normalized",
        as_index=False,
    )
    .agg(
        record_count=("instruction_id", "count"),
        scene_count=("scene_id", "nunique"),
    )
    .rename(
        columns={
            "flood_burden_normalized": "flood_burden",
        }
    )
)

disruption_distribution_df = (
    test_df.groupby(
        "disruption_normalized",
        as_index=False,
    )
    .agg(
        record_count=("instruction_id", "count"),
        scene_count=("scene_id", "nunique"),
    )
    .rename(
        columns={
            "disruption_normalized": "disruption_level",
        }
    )
)

print("\nTASK-FAMILY DISTRIBUTION")
print("-" * 80)
print(task_family_distribution_df.to_string(index=False))

print("\nRELIABILITY DISTRIBUTION")
print("-" * 80)
print(reliability_distribution_df.to_string(index=False))

print("\nFLOOD-BURDEN DISTRIBUTION")
print("-" * 80)
print(flood_burden_distribution_df.to_string(index=False))

print("\nDISRUPTION DISTRIBUTION")
print("-" * 80)
print(disruption_distribution_df.to_string(index=False))


# =============================================================================
# 9. VALIDATION CHECKS
# =============================================================================

expected_task_family_counts = (
    test_df.groupby(
        "task_family_normalized"
    )
    .size()
)

validation_checks = {
    "test_record_count_is_24": (
        len(test_df) == 24
    ),
    "test_scene_count_is_4": (
        test_df["scene_id"].nunique() == 4
    ),
    "task_family_count_is_6": (
        test_df[
            "task_family_normalized"
        ].nunique() == 6
    ),
    "four_records_per_task_family": bool(
        (expected_task_family_counts == 4).all()
    ),
    "six_records_per_scene": bool(
        (
            test_df.groupby("scene_id").size()
            == 6
        ).all()
    ),
    "instruction_ids_unique": bool(
        test_df["instruction_id"].is_unique
    ),
    "all_records_marked_test": bool(
        test_df["split"].eq("test").all()
    ),
    "all_records_training_ready": bool(
        test_df["training_ready"].all()
    ),
    "all_instruction_text_present": bool(
        test_df[
            "instruction_text"
        ].astype(str).str.strip().ne("").all()
    ),
    "all_reference_responses_present": bool(
        test_df[
            "response_text"
        ].astype(str).str.strip().ne("").all()
    ),
    "all_test_s2_images_valid": bool(
        image_validation_df[
            "scene_images_valid"
        ].all()
    ),
    "all_test_s1_images_valid": bool(
        image_validation_df[
            "scene_images_valid"
        ].all()
    ),
    "five_baselines_defined": (
        len(BASELINE_CONFIGURATIONS) == 5
    ),
}

validation_df = pd.DataFrame(
    {
        "validation_check": validation_checks.keys(),
        "passed": validation_checks.values(),
    }
)


# =============================================================================
# 10. SAVE CONFIGURATION AND VALIDATION OUTPUTS
# =============================================================================

BASELINE_CONFIGURATION_PATH = (
    CONFIG_DIR / "baseline_configurations.csv"
)

EXPERIMENT_CONFIGURATION_PATH = (
    CONFIG_DIR / "baseline_experiment_configuration.json"
)

TEST_MANIFEST_PATH = (
    CONFIG_DIR / "baseline_test_manifest.csv"
)

IMAGE_VALIDATION_PATH = (
    VALIDATION_DIR / "baseline_test_image_validation.csv"
)

SETUP_VALIDATION_PATH = (
    VALIDATION_DIR / "baseline_setup_validation.csv"
)

TASK_DISTRIBUTION_PATH = (
    VALIDATION_DIR / "baseline_task_distribution.csv"
)

RELIABILITY_DISTRIBUTION_PATH = (
    VALIDATION_DIR / "baseline_reliability_distribution.csv"
)

FLOOD_DISTRIBUTION_PATH = (
    VALIDATION_DIR / "baseline_flood_burden_distribution.csv"
)

DISRUPTION_DISTRIBUTION_PATH = (
    VALIDATION_DIR / "baseline_disruption_distribution.csv"
)

baseline_configuration_df.to_csv(
    BASELINE_CONFIGURATION_PATH,
    index=False,
)

test_df.to_csv(
    TEST_MANIFEST_PATH,
    index=False,
)

image_validation_df.to_csv(
    IMAGE_VALIDATION_PATH,
    index=False,
)

validation_df.to_csv(
    SETUP_VALIDATION_PATH,
    index=False,
)

task_family_distribution_df.to_csv(
    TASK_DISTRIBUTION_PATH,
    index=False,
)

reliability_distribution_df.to_csv(
    RELIABILITY_DISTRIBUTION_PATH,
    index=False,
)

flood_burden_distribution_df.to_csv(
    FLOOD_DISTRIBUTION_PATH,
    index=False,
)

disruption_distribution_df.to_csv(
    DISRUPTION_DISTRIBUTION_PATH,
    index=False,
)

experiment_configuration = {
    "notebook": "09_vlm_baselines",
    "experiment_name": "RoadFlood-VLM Baseline Evaluation",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "random_seed": RANDOM_SEED,
    "model_id": MODEL_ID,
    "device": DEVICE,
    "cuda_available": bool(CUDA_AVAILABLE),
    "gpu_name": GPU_NAME,
    "gpu_memory_gb": GPU_MEMORY_GB,
    "generation_configuration": GENERATION_CONFIG,
    "image_processing": {
        "minimum_pixels": MIN_PIXELS,
        "maximum_pixels": MAX_PIXELS,
    },
    "test_dataset": {
        "path": str(TEST_DATASET_PATH),
        "record_count": int(len(test_df)),
        "scene_count": int(
            test_df["scene_id"].nunique()
        ),
        "task_family_count": int(
            test_df[
                "task_family_normalized"
            ].nunique()
        ),
    },
    "baseline_configurations": (
        BASELINE_CONFIGURATIONS
    ),
    "package_versions": package_versions,
    "validation_results": {
        key: bool(value)
        for key, value in validation_checks.items()
    },
    "overall_status": (
        "complete"
        if all(validation_checks.values())
        else "failed_validation"
    ),
}

with open(
    EXPERIMENT_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        experiment_configuration,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# 11. FINAL REPORT
# =============================================================================

print("\nBASELINE SETUP VALIDATION")
print("-" * 80)

for check_name, passed in validation_checks.items():
    print(f"{check_name:<42}: {passed}")

print("\n" + "=" * 80)
print("NOTEBOOK 09 BASELINE SETUP COMPLETE")
print("=" * 80)

print(f"Model                       : {MODEL_ID}")
print(f"Selected device             : {DEVICE}")
print(f"Baseline configurations     : {len(BASELINE_CONFIGURATIONS)}")
print(f"Test records                : {len(test_df)}")
print(f"Test scenes                 : {test_df['scene_id'].nunique()}")
print(
    "Valid test image pairs      : "
    f"{image_validation_df['scene_images_valid'].sum()}"
)
print(
    "Overall setup status        : "
    f"{experiment_configuration['overall_status']}"
)

print("\nOUTPUT FILES")
print("-" * 80)

for output_path in [
    BASELINE_CONFIGURATION_PATH,
    EXPERIMENT_CONFIGURATION_PATH,
    TEST_MANIFEST_PATH,
    IMAGE_VALIDATION_PATH,
    SETUP_VALIDATION_PATH,
    TASK_DISTRIBUTION_PATH,
    RELIABILITY_DISTRIBUTION_PATH,
    FLOOD_DISTRIBUTION_PATH,
    DISRUPTION_DISTRIBUTION_PATH,
]:
    print(output_path)

if not all(validation_checks.values()):
    failed_checks = [
        check
        for check, passed in validation_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Notebook 09 setup failed validation: "
        + ", ".join(failed_checks)
    )

print("\nNEXT STEP")
print("-" * 80)
print(
    "Proceed to Notebook 09 Cell 2 to install or verify the "
    "Qwen2.5-VL inference dependencies and load the processor and model."
)

print("=" * 80)

NOTEBOOK 09: ROADFLOOD-VLM BASELINE EVALUATION

PROJECT PATHS
--------------------------------------------------------------------------------
Project root      : /home/adjeiowusu1/myproject/ResilientVLM
Test dataset      : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/vlm_dataset/splits/test.csv
Baseline outputs  : /home/adjeiowusu1/myproject/ResilientVLM/outputs/vlm_baselines

REPRODUCIBILITY
--------------------------------------------------------------------------------
Random seed : 42



SOFTWARE ENVIRONMENT
--------------------------------------------------------------------------------
python                  : 3.10.12
pandas                  : 2.3.3
numpy                   : 2.2.6
pillow                  : 12.3.0
torch                   : 2.13.0
transformers            : 5.14.1
accelerate              : 1.14.0
qwen-vl-utils           : 0.0.14
sentence-transformers   : 5.6.1
rouge-score             : 0.1.2
bert-score              : 0.3.13
evaluate                : 0.4.6

HARDWARE
--------------------------------------------------------------------------------
Operating system : Linux-6.8.0-111-generic-x86_64-with-glibc2.35
Processor        : x86_64
PyTorch detected : True
CUDA available   : False
Selected device  : cpu
GPU count        : 0
GPU name         : None
GPU memory       : 0.00 GB

MODEL CONFIGURATION
--------------------------------------------------------------------------------
Model ID       : Qwen/Qwen2.5-VL-3B-Instruct
Device         : cpu
Maximum tok

In [2]:
%pip install psutil

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [4]:
# =============================================================================
# NOTEBOOK 09 — CELL 2
# LOAD QWEN2.5-VL PROCESSOR AND MODEL
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import gc
import json
import os
import platform
import time
import traceback

import psutil
import torch
import transformers
from transformers import AutoProcessor, AutoModelForMultimodalLM


# =============================================================================
# 1. REQUIRED VARIABLES FROM CELL 1
# =============================================================================

required_variables = [
    "MODEL_ID",
    "MIN_PIXELS",
    "MAX_PIXELS",
    "CONFIG_DIR",
    "VALIDATION_DIR",
    "LOG_DIR",
    "DEVICE",
]

missing_variables = [
    variable_name
    for variable_name in required_variables
    if variable_name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run Notebook 09 Cell 1 before Cell 2. Missing variables: "
        + ", ".join(missing_variables)
    )


print("=" * 80)
print("NOTEBOOK 09: QWEN2.5-VL MODEL INITIALIZATION")
print("=" * 80)


# =============================================================================
# 2. SYSTEM MEMORY CHECK
# =============================================================================

memory = psutil.virtual_memory()

total_ram_gb = memory.total / (1024 ** 3)
available_ram_gb_before = memory.available / (1024 ** 3)
used_ram_percent_before = memory.percent

print("\nSYSTEM MEMORY BEFORE MODEL LOADING")
print("-" * 80)
print(f"Total system RAM     : {total_ram_gb:.2f} GB")
print(f"Available system RAM : {available_ram_gb_before:.2f} GB")
print(f"RAM utilization      : {used_ram_percent_before:.1f}%")

# A 3B-parameter model in float32 may require roughly 12 GB for weights alone,
# plus processor, vision encoder, temporary tensors, and generation overhead.
RECOMMENDED_AVAILABLE_RAM_GB = 16.0

if available_ram_gb_before < RECOMMENDED_AVAILABLE_RAM_GB:
    print(
        "\nWARNING: Available RAM is below the recommended "
        f"{RECOMMENDED_AVAILABLE_RAM_GB:.0f} GB."
    )
    print(
        "The model may load slowly, use virtual memory, or fail with "
        "an out-of-memory error."
    )


# =============================================================================
# 3. CPU THREAD CONFIGURATION
# =============================================================================

logical_cpu_count = os.cpu_count() or 1

# Leave some capacity for Windows and JupyterLab.
INFERENCE_CPU_THREADS = max(
    1,
    min(8, logical_cpu_count - 2),
)

torch.set_num_threads(INFERENCE_CPU_THREADS)

try:
    torch.set_num_interop_threads(
        max(1, min(4, INFERENCE_CPU_THREADS))
    )
except RuntimeError:
    # PyTorch may prevent changing interop threads after parallel work starts.
    pass

print("\nCPU CONFIGURATION")
print("-" * 80)
print(f"Logical CPU count       : {logical_cpu_count}")
print(f"PyTorch inference threads: {torch.get_num_threads()}")
print(f"Selected device         : {DEVICE}")


# =============================================================================
# 4. MODEL-LOADING CONFIGURATION
# =============================================================================

MODEL_DTYPE = torch.float32
LOW_CPU_MEMORY_USAGE = True
LOCAL_FILES_ONLY = False

model_loading_configuration = {
    "model_id": MODEL_ID,
    "device": "cpu",
    "torch_dtype": str(MODEL_DTYPE),
    "low_cpu_mem_usage": LOW_CPU_MEMORY_USAGE,
    "local_files_only": LOCAL_FILES_ONLY,
    "minimum_pixels": int(MIN_PIXELS),
    "maximum_pixels": int(MAX_PIXELS),
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "cpu_threads": int(torch.get_num_threads()),
}

print("\nMODEL-LOADING CONFIGURATION")
print("-" * 80)

for key, value in model_loading_configuration.items():
    print(f"{key:<24}: {value}")


# =============================================================================
# 5. LOAD PROCESSOR
# =============================================================================

processor_load_started = time.perf_counter()

print("\nLOADING PROCESSOR")
print("-" * 80)
print(f"Model repository: {MODEL_ID}")

try:
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
        local_files_only=LOCAL_FILES_ONLY,
    )

    processor_load_seconds = (
        time.perf_counter() - processor_load_started
    )

    processor_loaded = True
    processor_error = ""

    print("Processor loaded successfully.")
    print(
        f"Processor load time: "
        f"{processor_load_seconds:.2f} seconds"
    )

except Exception as exc:
    processor_loaded = False
    processor_error = traceback.format_exc()

    raise RuntimeError(
        "The Qwen2.5-VL processor could not be loaded.\n"
        f"Original error: {exc}"
    ) from exc


# =============================================================================
# 6. LOAD MODEL
# =============================================================================

gc.collect()

model_load_started = time.perf_counter()

print("\nLOADING MODEL")
print("-" * 80)
print(
    "This is the first large download and may take time. "
    "Do not interrupt the kernel while files are downloading."
)

try:
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=MODEL_DTYPE,
        low_cpu_mem_usage=LOW_CPU_MEMORY_USAGE,
        local_files_only=LOCAL_FILES_ONLY,
    )

    model.to("cpu")
    model.eval()

    model_load_seconds = (
        time.perf_counter() - model_load_started
    )

    model_loaded = True
    model_error = ""

    print("Model loaded successfully.")
    print(
        f"Model load time: "
        f"{model_load_seconds / 60:.2f} minutes"
    )

except Exception as exc:
    model_loaded = False
    model_error = traceback.format_exc()

    gc.collect()

    raise RuntimeError(
        "Qwen2.5-VL could not be loaded on the CPU.\n\n"
        "Likely causes include insufficient system RAM, an interrupted "
        "model download, or a Transformers compatibility issue.\n\n"
        f"Original error: {exc}"
    ) from exc


# =============================================================================
# 7. MODEL ARCHITECTURE AND PARAMETER CHECK
# =============================================================================

total_parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

model_parameter_size_gb = sum(
    parameter.numel() * parameter.element_size()
    for parameter in model.parameters()
) / (1024 ** 3)

model_device_set = sorted(
    {
        str(parameter.device)
        for parameter in model.parameters()
    }
)

model_dtype_set = sorted(
    {
        str(parameter.dtype)
        for parameter in model.parameters()
    }
)

print("\nMODEL INFORMATION")
print("-" * 80)
print(f"Model class          : {model.__class__.__name__}")
print(f"Total parameters     : {total_parameter_count:,}")
print(f"Trainable parameters : {trainable_parameter_count:,}")
print(f"Parameter memory     : {model_parameter_size_gb:.2f} GB")
print(f"Model device(s)      : {model_device_set}")
print(f"Model dtype(s)       : {model_dtype_set}")


# =============================================================================
# 8. SYSTEM MEMORY AFTER MODEL LOADING
# =============================================================================

memory_after = psutil.virtual_memory()

available_ram_gb_after = (
    memory_after.available / (1024 ** 3)
)

used_ram_percent_after = memory_after.percent

estimated_ram_consumed_gb = (
    available_ram_gb_before
    - available_ram_gb_after
)

print("\nSYSTEM MEMORY AFTER MODEL LOADING")
print("-" * 80)
print(
    f"Available system RAM : "
    f"{available_ram_gb_after:.2f} GB"
)
print(
    f"RAM utilization      : "
    f"{used_ram_percent_after:.1f}%"
)
print(
    f"Estimated RAM change : "
    f"{estimated_ram_consumed_gb:.2f} GB"
)


# =============================================================================
# 9. TEXT-ONLY SMOKE TEST
# =============================================================================

SMOKE_TEST_PROMPT = (
    "Respond with exactly this phrase: "
    "RoadFlood-VLM baseline model ready."
)

smoke_messages = [
    {
        "role": "system",
        "content": (
            "You are a concise transportation flood-risk "
            "assessment assistant."
        ),
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": SMOKE_TEST_PROMPT,
            }
        ],
    },
]

print("\nRUNNING TEXT-ONLY SMOKE TEST")
print("-" * 80)

smoke_test_started = time.perf_counter()

try:
    smoke_inputs = processor.apply_chat_template(
        smoke_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    smoke_inputs = {
        key: value.to("cpu")
        if hasattr(value, "to")
        else value
        for key, value in smoke_inputs.items()
    }

    input_token_count = int(
        smoke_inputs["input_ids"].shape[-1]
    )

    with torch.inference_mode():
        smoke_generated_ids = model.generate(
            **smoke_inputs,
            max_new_tokens=32,
            do_sample=False,
            use_cache=True,
        )

    generated_token_ids = smoke_generated_ids[
        :,
        input_token_count:
    ]

    smoke_response = processor.batch_decode(
        generated_token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    smoke_test_seconds = (
        time.perf_counter() - smoke_test_started
    )

    smoke_test_passed = bool(
        smoke_response
        and "RoadFlood-VLM" in smoke_response
    )

    smoke_test_error = ""

except Exception:
    smoke_test_seconds = (
        time.perf_counter() - smoke_test_started
    )

    smoke_response = ""
    smoke_test_passed = False
    smoke_test_error = traceback.format_exc()


print(f"Input tokens        : {input_token_count}")
print(f"Generated response  : {smoke_response}")
print(f"Generation time     : {smoke_test_seconds:.2f} seconds")
print(f"Smoke test passed   : {smoke_test_passed}")

if smoke_test_error:
    print("\nSMOKE TEST ERROR")
    print(smoke_test_error)


# =============================================================================
# 10. VALIDATION CHECKS
# =============================================================================

validation_checks = {
    "processor_loaded": bool(processor_loaded),
    "model_loaded": bool(model_loaded),
    "model_is_in_evaluation_mode": bool(
        model.training is False
    ),
    "model_parameters_detected": bool(
        total_parameter_count > 0
    ),
    "model_on_cpu": bool(
        model_device_set == ["cpu"]
    ),
    "processor_has_chat_template": bool(
        getattr(processor, "chat_template", None)
    ),
    "smoke_test_generated_text": bool(
        smoke_response.strip()
    ),
    "smoke_test_passed": bool(
        smoke_test_passed
    ),
}

model_validation_df = pd.DataFrame(
    {
        "validation_check": validation_checks.keys(),
        "passed": validation_checks.values(),
    }
)


# =============================================================================
# 11. SAVE OUTPUTS
# =============================================================================

MODEL_CONFIGURATION_PATH = (
    CONFIG_DIR
    / "qwen25_vl_cpu_model_configuration.json"
)

MODEL_VALIDATION_PATH = (
    VALIDATION_DIR
    / "qwen25_vl_model_validation.csv"
)

MODEL_INITIALIZATION_LOG_PATH = (
    LOG_DIR
    / "qwen25_vl_model_initialization.json"
)

model_validation_df.to_csv(
    MODEL_VALIDATION_PATH,
    index=False,
)

model_initialization_log = {
    "notebook": "09_vlm_baselines",
    "cell": 2,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "model_id": MODEL_ID,
    "model_class": model.__class__.__name__,
    "processor_class": processor.__class__.__name__,
    "device": "cpu",
    "model_dtype_set": model_dtype_set,
    "model_device_set": model_device_set,
    "total_parameter_count": int(
        total_parameter_count
    ),
    "trainable_parameter_count": int(
        trainable_parameter_count
    ),
    "model_parameter_size_gb": float(
        model_parameter_size_gb
    ),
    "processor_load_seconds": float(
        processor_load_seconds
    ),
    "model_load_seconds": float(
        model_load_seconds
    ),
    "smoke_test_seconds": float(
        smoke_test_seconds
    ),
    "smoke_test_prompt": SMOKE_TEST_PROMPT,
    "smoke_test_response": smoke_response,
    "available_ram_gb_before": float(
        available_ram_gb_before
    ),
    "available_ram_gb_after": float(
        available_ram_gb_after
    ),
    "estimated_ram_consumed_gb": float(
        estimated_ram_consumed_gb
    ),
    "validation_results": {
        key: bool(value)
        for key, value in validation_checks.items()
    },
    "overall_status": (
        "complete"
        if all(validation_checks.values())
        else "failed_validation"
    ),
}

with open(
    MODEL_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_loading_configuration,
        file,
        indent=2,
        ensure_ascii=False,
    )

with open(
    MODEL_INITIALIZATION_LOG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_initialization_log,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# 12. FINAL REPORT
# =============================================================================

print("\nMODEL INITIALIZATION VALIDATION")
print("-" * 80)

for check_name, passed in validation_checks.items():
    print(f"{check_name:<40}: {passed}")

print("\n" + "=" * 80)
print("NOTEBOOK 09 CELL 2 COMPLETE")
print("=" * 80)

print(f"Processor class       : {processor.__class__.__name__}")
print(f"Model class           : {model.__class__.__name__}")
print(f"Model parameters      : {total_parameter_count:,}")
print(f"Parameter memory      : {model_parameter_size_gb:.2f} GB")
print(f"Model device          : CPU")
print(f"Model load time       : {model_load_seconds / 60:.2f} minutes")
print(f"Smoke-test time       : {smoke_test_seconds:.2f} seconds")
print(
    "Overall status        : "
    f"{model_initialization_log['overall_status']}"
)

print("\nOUTPUT FILES")
print("-" * 80)
print(MODEL_CONFIGURATION_PATH)
print(MODEL_VALIDATION_PATH)
print(MODEL_INITIALIZATION_LOG_PATH)

if not all(validation_checks.values()):
    failed_checks = [
        check_name
        for check_name, passed
        in validation_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Notebook 09 Cell 2 failed validation: "
        + ", ".join(failed_checks)
    )

print("\nNEXT STEP")
print("-" * 80)
print(
    "Proceed to Notebook 09 Cell 3 to define multimodal prompt "
    "construction, structured transportation context, and a "
    "single-record inference test."
)

print("=" * 80)

NOTEBOOK 09: QWEN2.5-VL MODEL INITIALIZATION

SYSTEM MEMORY BEFORE MODEL LOADING
--------------------------------------------------------------------------------
Total system RAM     : 125.57 GB
Available system RAM : 107.81 GB
RAM utilization      : 14.1%

CPU CONFIGURATION
--------------------------------------------------------------------------------
Logical CPU count       : 80
PyTorch inference threads: 8
Selected device         : cpu

MODEL-LOADING CONFIGURATION
--------------------------------------------------------------------------------
model_id                : Qwen/Qwen2.5-VL-3B-Instruct
device                  : cpu
torch_dtype             : torch.float32
low_cpu_mem_usage       : True
local_files_only        : False
minimum_pixels          : 200704
maximum_pixels          : 401408
transformers_version    : 5.14.1
torch_version           : 2.13.0+cu130
cpu_threads             : 8

LOADING PROCESSOR
-------------------------------------------------------------------------

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Processor loaded successfully.
Processor load time: 1.39 seconds

LOADING MODEL
--------------------------------------------------------------------------------
This is the first large download and may take time. Do not interrupt the kernel while files are downloading.


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Model loaded successfully.
Model load time: 0.03 minutes

MODEL INFORMATION
--------------------------------------------------------------------------------
Model class          : Qwen2_5_VLForConditionalGeneration
Total parameters     : 3,754,622,976
Trainable parameters : 3,754,622,976
Parameter memory     : 13.99 GB
Model device(s)      : ['cpu']
Model dtype(s)       : ['torch.float32']

SYSTEM MEMORY AFTER MODEL LOADING
--------------------------------------------------------------------------------
Available system RAM : 93.65 GB
RAM utilization      : 25.4%
Estimated RAM change : 14.15 GB

RUNNING TEXT-ONLY SMOKE TEST
--------------------------------------------------------------------------------


Input tokens        : 38
Generated response  : RoadFlood-VLM baseline model ready.
Generation time     : 3.66 seconds
Smoke test passed   : True

MODEL INITIALIZATION VALIDATION
--------------------------------------------------------------------------------
processor_loaded                        : True
model_loaded                            : True
model_is_in_evaluation_mode             : True
model_parameters_detected               : True
model_on_cpu                            : True
processor_has_chat_template             : True
smoke_test_generated_text               : True
smoke_test_passed                       : True

NOTEBOOK 09 CELL 2 COMPLETE
Processor class       : Qwen2_5_VLProcessor
Model class           : Qwen2_5_VLForConditionalGeneration
Model parameters      : 3,754,622,976
Parameter memory      : 13.99 GB
Model device          : CPU
Model load time       : 0.03 minutes
Smoke-test time       : 3.66 seconds
Overall status        : complete

OUTPUT FILES
-------------

In [5]:
# =============================================================================
# NOTEBOOK 09 — CELL 3
# MULTIMODAL PROMPT CONSTRUCTION AND SINGLE-RECORD INFERENCE TEST
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import gc
import json
import time
import traceback

import pandas as pd
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info


# =============================================================================
# 1. VERIFY REQUIRED OBJECTS FROM CELLS 1 AND 2
# =============================================================================

required_objects = [
    "processor",
    "model",
    "test_df",
    "BASELINE_CONFIGURATIONS",
    "MODEL_ID",
    "OUTPUT_ROOT",
    "VALIDATION_DIR",
    "LOG_DIR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run Notebook 09 Cells 1 and 2 before Cell 3. "
        "Missing objects: "
        + ", ".join(missing_objects)
    )


print("=" * 80)
print("NOTEBOOK 09: MULTIMODAL PROMPT AND SINGLE-RECORD TEST")
print("=" * 80)


# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def first_existing_column(
    dataframe: pd.DataFrame,
    candidate_columns: list[str],
    required: bool = True,
) -> str | None:
    """
    Return the first matching column name from a list of candidates.
    """

    for column_name in candidate_columns:
        if column_name in dataframe.columns:
            return column_name

    if required:
        raise KeyError(
            "None of the expected columns were found: "
            + ", ".join(candidate_columns)
        )

    return None


def clean_value(value) -> str:
    """
    Convert a value to clean text while handling missing values.
    """

    if pd.isna(value):
        return ""

    return str(value).strip()


def resolve_existing_path(
    raw_path,
    project_root: Path | None = None,
) -> Path:
    """
    Resolve an image path stored as an absolute or project-relative path.
    """

    path_text = clean_value(raw_path)

    if not path_text:
        raise ValueError("The image path is empty.")

    candidate_path = Path(path_text)

    if candidate_path.exists():
        return candidate_path.resolve()

    if project_root is not None:
        project_candidate = project_root / candidate_path

        if project_candidate.exists():
            return project_candidate.resolve()

    raise FileNotFoundError(
        f"Image file was not found: {path_text}"
    )


def path_to_file_uri(path: Path) -> str:
    """
    Convert a local Windows or Unix path to a file URI.
    """

    return path.resolve().as_uri()


def count_image_pixels(path: Path) -> tuple[int, int, int]:
    """
    Return image width, height, and total pixel count.
    """

    with Image.open(path) as image:
        width, height = image.size

    return width, height, width * height


# =============================================================================
# 3. IDENTIFY DATASET COLUMNS
# =============================================================================

instruction_column = first_existing_column(
    test_df,
    [
        "instruction",
        "instruction_text",
        "prompt",
        "question",
        "user_prompt",
    ],
)

reference_column = first_existing_column(
    test_df,
    [
        "reference_response",
        "reference_answer",
        "response",
        "answer",
        "target_response",
        "ground_truth",
    ],
)

scene_column = first_existing_column(
    test_df,
    [
        "scene_id",
        "scene",
        "scene_name",
    ],
)

task_family_column = first_existing_column(
    test_df,
    [
        "task_family",
        "task_type",
        "task",
    ],
)

instruction_id_column = first_existing_column(
    test_df,
    [
        "instruction_id",
        "record_id",
        "sample_id",
        "id",
    ],
)

s2_path_column = first_existing_column(
    test_df,
    [
        "s2_png_path",
        "training_s2_path",
        "s2_png_relative_path",
        "training_s2_relative_path",
        "sentinel2_image_path",
        "sentinel_2_image_path",
        "s2_image_path",
        "sentinel2_path",
        "s2_path",
        "optical_image_path",
    ],
)

s1_path_column = first_existing_column(
    test_df,
    [
        "s1_png_path",
        "training_s1_path",
        "s1_png_relative_path",
        "training_s1_relative_path",
        "sentinel1_image_path",
        "sentinel_1_image_path",
        "s1_image_path",
        "sentinel1_path",
        "s1_path",
        "sar_image_path",
    ],
)

PROJECT_ROOT_CELL3 = (
    Path(PROJECT_ROOT)
    if "PROJECT_ROOT" in globals()
    else Path.cwd()
)


print("\nDETECTED DATASET COLUMNS")
print("-" * 80)
print(f"Instruction ID      : {instruction_id_column}")
print(f"Scene ID            : {scene_column}")
print(f"Task family         : {task_family_column}")
print(f"Instruction         : {instruction_column}")
print(f"Reference response  : {reference_column}")
print(f"Sentinel-2 path     : {s2_path_column}")
print(f"Sentinel-1 path     : {s1_path_column}")


# =============================================================================
# 4. STRUCTURED TRANSPORTATION-CONTEXT BUILDER
# =============================================================================

CONTEXT_COLUMN_GROUPS = {
    "Flood evidence": [
        "flood_burden",
        "flood_extent",
        "flood_percentage",
        "flood_pct",
        "flooded_area_pct",
        "water_coverage",
    ],
    "Reliability": [
        "reliability",
        "reliability_level",
        "confidence",
        "confidence_level",
        "valid_pct",
        "ignore_pct",
    ],
    "Transportation disruption": [
        "disruption_level",
        "transportation_disruption",
        "network_disruption",
        "road_disruption",
    ],
    "Road-network exposure": [
        "road_record_count",
        "road_count",
        "major_road_count",
        "critical_transport_edge_count",
        "critical_network_edge_count",
        "single_access_road_count",
    ],
    "Network vulnerability": [
        "critical_low_redundancy_count",
        "bridge_bottleneck_count",
        "major_road_bottleneck_count",
        "high_topological_vulnerability_count",
        "articulation_point_count",
        "graph_bridge_record_count",
    ],
    "Connectivity and redundancy": [
        "connected_components",
        "largest_component_share",
        "independent_cycle_count",
        "roads_with_alternate_routes",
        "network_redundancy",
    ],
    "Location": [
        "country",
        "region",
        "location",
    ],
}


def build_transportation_context(
    record: pd.Series,
) -> str:
    """
    Build a structured context block using fields available in the dataset.
    """

    context_lines = []

    for group_name, candidate_columns in CONTEXT_COLUMN_GROUPS.items():
        group_items = []

        for column_name in candidate_columns:
            if column_name not in record.index:
                continue

            value = record[column_name]

            if pd.isna(value):
                continue

            value_text = clean_value(value)

            if not value_text:
                continue

            readable_name = (
                column_name
                .replace("_", " ")
                .strip()
                .title()
            )

            group_items.append(
                f"- {readable_name}: {value_text}"
            )

        if group_items:
            context_lines.append(f"{group_name}:")
            context_lines.extend(group_items)

    if not context_lines:
        return (
            "No additional structured transportation-context "
            "variables are available for this record."
        )

    return "\n".join(context_lines)


# =============================================================================
# 5. BASELINE PROMPT BUILDER
# =============================================================================

SYSTEM_PROMPT = """
You are RoadFlood-VLM, a transportation flood-risk assessment assistant.

Use only the evidence provided in the prompt and images. Distinguish direct
visual evidence from structured contextual evidence. Do not invent road names,
flood depths, closure conditions, or infrastructure impacts that are not
supported by the supplied evidence.

Respond directly to the requested task. Keep the response concise, technically
grounded, and suitable for transportation-resilience analysis.
""".strip()


IMAGE_DESCRIPTIONS = {
    "sentinel2": (
        "Sentinel-2 optical image. Use visible surface, water, land-cover, "
        "and roadway-pattern evidence where discernible."
    ),
    "sentinel1": (
        "Sentinel-1 SAR visualization. Use radar-derived flood and surface "
        "contrast evidence cautiously."
    ),
}


def build_baseline_messages(
    record: pd.Series,
    baseline_id: str,
) -> tuple[list[dict], dict]:
    """
    Construct Qwen messages for one record and one baseline.
    """

    baseline_lookup = BASELINE_CONFIGURATIONS

    if baseline_id not in baseline_lookup:
        raise KeyError(
            f"Unknown baseline ID: {baseline_id}"
        )

    baseline = baseline_lookup[baseline_id]

    instruction_text = clean_value(
        record[instruction_column]
    )

    if not instruction_text:
        raise ValueError(
            "The selected record has no instruction text."
        )

    user_content = []
    image_paths = []

    if bool(baseline["use_sentinel2"]):
        s2_path = resolve_existing_path(
            record[s2_path_column],
            project_root=PROJECT_ROOT_CELL3,
        )

        user_content.extend(
            [
                {
                    "type": "text",
                    "text": (
                        "Image 1 description: "
                        + IMAGE_DESCRIPTIONS["sentinel2"]
                    ),
                },
                {
                    "type": "image",
                    "image": path_to_file_uri(s2_path),
                },
            ]
        )

        image_paths.append(str(s2_path))

    if bool(baseline["use_sentinel1"]):
        s1_path = resolve_existing_path(
            record[s1_path_column],
            project_root=PROJECT_ROOT_CELL3,
        )

        image_number = len(image_paths) + 1

        user_content.extend(
            [
                {
                    "type": "text",
                    "text": (
                        f"Image {image_number} description: "
                        + IMAGE_DESCRIPTIONS["sentinel1"]
                    ),
                },
                {
                    "type": "image",
                    "image": path_to_file_uri(s1_path),
                },
            ]
        )

        image_paths.append(str(s1_path))

    prompt_sections = [
        "TASK",
        instruction_text,
    ]

    if bool(baseline["use_structured_context"]):
        transportation_context = (
            build_transportation_context(record)
        )

        prompt_sections.extend(
            [
                "",
                "STRUCTURED TRANSPORTATION CONTEXT",
                transportation_context,
            ]
        )
    else:
        transportation_context = ""

    prompt_sections.extend(
        [
            "",
            "RESPONSE REQUIREMENTS",
            (
                "Answer the task using only the supplied evidence. "
                "State uncertainty when the evidence is insufficient."
            ),
        ]
    )

    user_content.append(
        {
            "type": "text",
            "text": "\n".join(prompt_sections),
        }
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_content,
        },
    ]

    metadata = {
        "baseline_id": baseline_id,
        "baseline_name": baseline["display_name"],
        "use_sentinel2": bool(
            baseline["use_sentinel2"]
        ),
        "use_sentinel1": bool(
            baseline["use_sentinel1"]
        ),
        "use_structured_context": bool(
            baseline["use_structured_context"]
        ),
        "image_paths": image_paths,
        "image_count": len(image_paths),
        "transportation_context": transportation_context,
        "instruction_text": instruction_text,
    }

    return messages, metadata


# =============================================================================
# 6. SINGLE-RECORD INFERENCE FUNCTION
# =============================================================================

def run_single_inference(
    record: pd.Series,
    baseline_id: str,
    max_new_tokens: int = 64,
) -> dict:
    """
    Run one deterministic Qwen2.5-VL inference.
    """

    messages, prompt_metadata = (
        build_baseline_messages(
            record=record,
            baseline_id=baseline_id,
        )
    )

    formatted_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(
        messages
    )

    inference_inputs = processor(
        text=[formatted_text],
        images=image_inputs or None,
        videos=video_inputs or None,
        padding=True,
        return_tensors="pt",
    )

    inference_inputs = {
        key: (
            value.to("cpu")
            if hasattr(value, "to")
            else value
        )
        for key, value in inference_inputs.items()
    }

    input_token_count = int(
        inference_inputs["input_ids"].shape[-1]
    )

    pixel_value_shape = None

    if "pixel_values" in inference_inputs:
        pixel_value_shape = list(
            inference_inputs["pixel_values"].shape
        )

    gc.collect()

    inference_started = time.perf_counter()

    with torch.inference_mode():
        generated_ids = model.generate(
            **inference_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    inference_seconds = (
        time.perf_counter() - inference_started
    )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids
        in zip(
            inference_inputs["input_ids"],
            generated_ids,
        )
    ]

    generated_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    generated_token_count = int(
        generated_ids_trimmed[0].shape[-1]
    )

    result = {
        **prompt_metadata,
        "instruction_id": clean_value(
            record[instruction_id_column]
        ),
        "scene_id": clean_value(
            record[scene_column]
        ),
        "task_family": clean_value(
            record[task_family_column]
        ),
        "reference_response": clean_value(
            record[reference_column]
        ),
        "generated_response": generated_text,
        "input_token_count": input_token_count,
        "generated_token_count": generated_token_count,
        "max_new_tokens": max_new_tokens,
        "pixel_value_shape": pixel_value_shape,
        "inference_seconds": float(
            inference_seconds
        ),
        "inference_minutes": float(
            inference_seconds / 60
        ),
        "status": (
            "complete"
            if generated_text
            else "empty_response"
        ),
    }

    del inference_inputs
    del generated_ids
    del generated_ids_trimmed
    del image_inputs
    del video_inputs

    gc.collect()

    return result


# =============================================================================
# 7. SELECT ONE TEST RECORD
# =============================================================================

# Start with the Sentinel-2-only baseline to reduce memory use.
SINGLE_TEST_BASELINE_ID = "B1_s2_only"
SINGLE_TEST_MAX_NEW_TOKENS = 64

# Select the first valid test record.
single_test_record = test_df.iloc[0].copy()

single_test_s2_path = resolve_existing_path(
    single_test_record[s2_path_column],
    project_root=PROJECT_ROOT_CELL3,
)

s2_width, s2_height, s2_pixels = count_image_pixels(
    single_test_s2_path
)


print("\nSINGLE-RECORD TEST CONFIGURATION")
print("-" * 80)
print(
    f"Instruction ID : "
    f"{clean_value(single_test_record[instruction_id_column])}"
)
print(
    f"Scene ID       : "
    f"{clean_value(single_test_record[scene_column])}"
)
print(
    f"Task family    : "
    f"{clean_value(single_test_record[task_family_column])}"
)
print(f"Baseline       : {SINGLE_TEST_BASELINE_ID}")
print(
    f"Maximum tokens : "
    f"{SINGLE_TEST_MAX_NEW_TOKENS}"
)
print(f"Sentinel-2     : {single_test_s2_path}")
print(
    f"Original size  : "
    f"{s2_width} × {s2_height} "
    f"({s2_pixels:,} pixels)"
)


# =============================================================================
# 8. PREVIEW THE CONSTRUCTED PROMPT
# =============================================================================

preview_messages, preview_metadata = (
    build_baseline_messages(
        record=single_test_record,
        baseline_id=SINGLE_TEST_BASELINE_ID,
    )
)

preview_text = processor.apply_chat_template(
    preview_messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("\nPROMPT PREVIEW")
print("-" * 80)
print(preview_text[:3000])

if len(preview_text) > 3000:
    print("\n[Prompt preview truncated]")


# =============================================================================
# 9. RUN THE SINGLE MULTIMODAL INFERENCE
# =============================================================================

print("\nRUNNING SINGLE MULTIMODAL INFERENCE")
print("-" * 80)
print(
    "This CPU inference may take several minutes. "
    "Do not rerun the cell while it is processing."
)

single_test_started = time.perf_counter()

try:
    single_test_result = run_single_inference(
        record=single_test_record,
        baseline_id=SINGLE_TEST_BASELINE_ID,
        max_new_tokens=SINGLE_TEST_MAX_NEW_TOKENS,
    )

    single_test_error = ""
    single_test_passed = bool(
        single_test_result["generated_response"]
    )

except Exception as exc:
    single_test_result = {}
    single_test_error = traceback.format_exc()
    single_test_passed = False

    print("\nINFERENCE ERROR")
    print(single_test_error)

    raise RuntimeError(
        "The single-record multimodal inference failed. "
        f"Original error: {exc}"
    ) from exc


# =============================================================================
# 10. DISPLAY RESULTS
# =============================================================================

print("\nSINGLE-RECORD INFERENCE RESULT")
print("-" * 80)

print(
    f"Instruction ID      : "
    f"{single_test_result['instruction_id']}"
)
print(
    f"Scene ID            : "
    f"{single_test_result['scene_id']}"
)
print(
    f"Task family         : "
    f"{single_test_result['task_family']}"
)
print(
    f"Baseline            : "
    f"{single_test_result['baseline_name']}"
)
print(
    f"Input tokens        : "
    f"{single_test_result['input_token_count']}"
)
print(
    f"Generated tokens    : "
    f"{single_test_result['generated_token_count']}"
)
print(
    f"Vision tensor shape : "
    f"{single_test_result['pixel_value_shape']}"
)
print(
    f"Inference time      : "
    f"{single_test_result['inference_minutes']:.2f} minutes"
)
print(
    f"Status              : "
    f"{single_test_result['status']}"
)

print("\nINSTRUCTION")
print("-" * 80)
print(single_test_result["instruction_text"])

print("\nMODEL RESPONSE")
print("-" * 80)
print(single_test_result["generated_response"])

print("\nREFERENCE RESPONSE")
print("-" * 80)
print(single_test_result["reference_response"])


# =============================================================================
# 11. VALIDATION
# =============================================================================

cell3_validation = {
    "baseline_configuration_found": (
        SINGLE_TEST_BASELINE_ID
        in BASELINE_CONFIGURATIONS
    ),
    "sentinel2_image_exists": (
        single_test_s2_path.exists()
    ),
    "messages_constructed": bool(
        preview_messages
    ),
    "prompt_constructed": bool(
        preview_text.strip()
    ),
    "single_inference_completed": bool(
        single_test_passed
    ),
    "generated_response_present": bool(
        single_test_result[
            "generated_response"
        ].strip()
    ),
    "input_tokens_detected": bool(
        single_test_result[
            "input_token_count"
        ] > 0
    ),
    "vision_tensor_detected": bool(
        single_test_result[
            "pixel_value_shape"
        ] is not None
    ),
}

cell3_validation_df = pd.DataFrame(
    {
        "validation_check": (
            cell3_validation.keys()
        ),
        "passed": (
            cell3_validation.values()
        ),
    }
)


# =============================================================================
# 12. SAVE OUTPUTS
# =============================================================================

SINGLE_TEST_OUTPUT_DIR = (
    OUTPUT_ROOT / "single_record_test"
)

SINGLE_TEST_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SINGLE_TEST_RESULT_PATH = (
    SINGLE_TEST_OUTPUT_DIR
    / "cell3_single_record_inference.json"
)

SINGLE_TEST_RESULT_CSV_PATH = (
    SINGLE_TEST_OUTPUT_DIR
    / "cell3_single_record_inference.csv"
)

CELL3_VALIDATION_PATH = (
    VALIDATION_DIR
    / "cell3_single_record_validation.csv"
)

CELL3_LOG_PATH = (
    LOG_DIR
    / "cell3_single_record_inference_log.json"
)


serializable_result = {
    key: value
    for key, value in single_test_result.items()
}

with open(
    SINGLE_TEST_RESULT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        serializable_result,
        file,
        indent=2,
        ensure_ascii=False,
    )

pd.DataFrame(
    [serializable_result]
).to_csv(
    SINGLE_TEST_RESULT_CSV_PATH,
    index=False,
)

cell3_validation_df.to_csv(
    CELL3_VALIDATION_PATH,
    index=False,
)

cell3_log = {
    "notebook": "09_vlm_baselines",
    "cell": 3,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "baseline_id": SINGLE_TEST_BASELINE_ID,
    "instruction_id": (
        single_test_result["instruction_id"]
    ),
    "scene_id": (
        single_test_result["scene_id"]
    ),
    "task_family": (
        single_test_result["task_family"]
    ),
    "image_count": (
        single_test_result["image_count"]
    ),
    "input_token_count": (
        single_test_result["input_token_count"]
    ),
    "generated_token_count": (
        single_test_result[
            "generated_token_count"
        ]
    ),
    "inference_seconds": (
        single_test_result[
            "inference_seconds"
        ]
    ),
    "validation_results": {
        key: bool(value)
        for key, value
        in cell3_validation.items()
    },
    "overall_status": (
        "complete"
        if all(cell3_validation.values())
        else "failed_validation"
    ),
}

with open(
    CELL3_LOG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        cell3_log,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# 13. FINAL REPORT
# =============================================================================

print("\nCELL 3 VALIDATION")
print("-" * 80)

for check_name, passed in cell3_validation.items():
    print(f"{check_name:<40}: {passed}")

print("\n" + "=" * 80)
print("NOTEBOOK 09 CELL 3 COMPLETE")
print("=" * 80)

print(
    f"Baseline tested       : "
    f"{single_test_result['baseline_name']}"
)
print(
    f"Instruction tested    : "
    f"{single_test_result['instruction_id']}"
)
print(
    f"Image count           : "
    f"{single_test_result['image_count']}"
)
print(
    f"Inference time        : "
    f"{single_test_result['inference_minutes']:.2f} minutes"
)
print(
    f"Generated tokens      : "
    f"{single_test_result['generated_token_count']}"
)
print(
    f"Overall status        : "
    f"{cell3_log['overall_status']}"
)

print("\nOUTPUT FILES")
print("-" * 80)
print(SINGLE_TEST_RESULT_PATH)
print(SINGLE_TEST_RESULT_CSV_PATH)
print(CELL3_VALIDATION_PATH)
print(CELL3_LOG_PATH)

if not all(cell3_validation.values()):
    failed_checks = [
        check_name
        for check_name, passed
        in cell3_validation.items()
        if not passed
    ]

    raise RuntimeError(
        "Notebook 09 Cell 3 failed validation: "
        + ", ".join(failed_checks)
    )

print("\nNEXT STEP")
print("-" * 80)
print(
    "Proceed to Cell 4 to test all five baseline prompt "
    "configurations on one record before launching the full evaluation."
)

print("=" * 80)

NOTEBOOK 09: MULTIMODAL PROMPT AND SINGLE-RECORD TEST

DETECTED DATASET COLUMNS
--------------------------------------------------------------------------------
Instruction ID      : instruction_id
Scene ID            : scene_id
Task family         : task_family
Instruction         : instruction
Reference response  : response
Sentinel-2 path     : s2_png_path
Sentinel-1 path     : s1_png_path

SINGLE-RECORD TEST CONFIGURATION
--------------------------------------------------------------------------------
Instruction ID : instruction_c87003dfcac8
Scene ID       : Mekong_16233
Task family    : scene_flood_assessment
Baseline       : B1_s2_only
Maximum tokens : 64
Sentinel-2     : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/vlm_dataset/images/sentinel2_rgb/Mekong_16233_S2_RGB.png
Original size  : 512 × 512 (262,144 pixels)

PROMPT PREVIEW
--------------------------------------------------------------------------------
<|im_start|>system
You are RoadFlood-VLM, a transportation


SINGLE-RECORD INFERENCE RESULT
--------------------------------------------------------------------------------
Instruction ID      : instruction_c87003dfcac8
Scene ID            : Mekong_16233
Task family         : scene_flood_assessment
Baseline            : Sentinel-2 only
Input tokens        : 515
Generated tokens    : 51
Vision tensor shape : [1296, 1176]
Inference time      : 0.36 minutes
Status              : complete

INSTRUCTION
--------------------------------------------------------------------------------
Assess the overall roadway flood burden in this scene. Use the grounded physical-road-edge counts and flood-exposure classes, and state the assigned scene flood-burden category.

MODEL RESPONSE
--------------------------------------------------------------------------------
The image shows an aerial view of a rural area with fields and roads. The visible flood exposure classes include areas with water covering parts of the roads and fields. Based on the visible flooding, 